# Business Analysis & KPI Generation

## Objective

Convert the Customer 360 dataset into business-level KPIs, segment performance metrics, actionable findings, and recommendations.

## Analysis Areas

* Overall banking KPIs
* Customer and financial metrics
* Segment-level performance
* Customer engagement
* Product usage
* Business findings
* Business recommendations
* Submission-ready outputs


In [1]:
import pandas as pd

customer_360 = pd.read_csv("../data/final/customer_360.csv")

print("Customer 360 loaded successfully")
print("Rows:", customer_360.shape[0])
print("Columns:", customer_360.shape[1])

Customer 360 loaded successfully
Rows: 5000
Columns: 29


In [2]:
print("Customer 360 columns:")
print("-" * 40)

for i, col in enumerate(customer_360.columns, start=1):
    print(f"{i}. {col}")

Customer 360 columns:
----------------------------------------
1. customer_id
2. age
3. gender
4. country
5. occupation
6. annual_income
7. credit_score
8. tenure_years
9. join_date
10. last_transaction_date
11. late_loan_count
12. default_loan_count
13. complaint_count
14. escalated_count
15. total_balance
16. active_account_count
17. recency_days
18. products_number
19. churn
20. loan_to_income_ratio
21. account_count
22. total_transactions
23. total_transaction_value
24. avg_transaction_value
25. loan_count
26. total_loan_amount
27. average_interest_rate
28. total_service_interactions
29. average_resolution_days


In [3]:
# Overall KPI
total_customers = customer_360["customer_id"].nunique()

total_deposits = customer_360["total_balance"].sum()

total_loan_amount = customer_360["total_loan_amount"].sum()

total_transactions = customer_360["total_transactions"].sum()

total_transaction_value = customer_360["total_transaction_value"].sum()

average_balance = customer_360["total_balance"].mean()

average_loan_amount = customer_360.loc[
    customer_360["loan_count"] > 0,
    "total_loan_amount"
].mean()

average_transaction_value = customer_360["avg_transaction_value"].mean()

loan_customers = (customer_360["loan_count"] > 0).sum()

active_customers = (customer_360["active_account_count"] > 0).sum()

loan_penetration = (loan_customers / total_customers) * 100

active_customer_rate = (active_customers / total_customers) * 100

print("OVERALL BANKING KPIs")
print("=" * 50)

print(f"Total Customers: {total_customers:,}")
print(f"Total Deposits / Balance: {total_deposits:,.2f}")
print(f"Total Loan Amount: {total_loan_amount:,.2f}")
print(f"Total Transactions: {total_transactions:,.0f}")
print(f"Total Transaction Value: {total_transaction_value:,.2f}")
print(f"Average Customer Balance: {average_balance:,.2f}")
print(f"Average Loan Amount: {average_loan_amount:,.2f}")
print(f"Average Transaction Value: {average_transaction_value:,.2f}")
print(f"Loan Customers: {loan_customers:,}")
print(f"Active Customers: {active_customers:,}")
print(f"Loan Penetration: {loan_penetration:.2f}%")
print(f"Active Customer Rate: {active_customer_rate:.2f}%")

OVERALL BANKING KPIs
Total Customers: 5,000
Total Deposits / Balance: 429,675,148.55
Total Loan Amount: 736,761,689.61
Total Transactions: 77,407
Total Transaction Value: 22,906,463.21
Average Customer Balance: 85,935.03
Average Loan Amount: 485,030.74
Average Transaction Value: 229.40
Loan Customers: 1,519
Active Customers: 3,644
Loan Penetration: 30.38%
Active Customer Rate: 72.88%


In [4]:
# Check whether the segmentation notebook saved a segmented file
import os

print("Files in data/final:")
print("-" * 40)

for file in os.listdir("../data/final"):
    print(file)

Files in data/final:
----------------------------------------
customer_360.csv
customer_360_segmented.csv


In [5]:
# Load the segmented Customer 360
customer_360_segmented = pd.read_csv(
    "../data/final/customer_360_segmented.csv"
)

print("Segmented Customer 360 loaded successfully")
print("Rows:", customer_360_segmented.shape[0])
print("Columns:", customer_360_segmented.shape[1])

Segmented Customer 360 loaded successfully
Rows: 5000
Columns: 31


In [6]:
print("\nSegment columns:")
print("-" * 40)

print(
    customer_360_segmented[
        ["customer_id", "segment", "segment_name"]
    ].head()
)


Segment columns:
----------------------------------------
  customer_id  segment            segment_name
0      C00001        1  Core Banking Customers
1      C00002        1  Core Banking Customers
2      C00003        1  Core Banking Customers
3      C00004        1  Core Banking Customers
4      C00005        1  Core Banking Customers


In [7]:
print("\nSegment distribution:")
print("-" * 40)

print(
    customer_360_segmented["segment_name"].value_counts()
)


Segment distribution:
----------------------------------------
segment_name
Core Banking Customers       2295
High-Service Customers       1167
Low Engagement Customers     1128
High-Value Loan Customers     410
Name: count, dtype: int64


In [11]:
# Create Segment-Level KPIs
segment_kpis = (
    customer_360_segmented
    .groupby("segment_name")
    .agg(
        customers=("customer_id", "nunique"),
        total_balance=("total_balance", "sum"),
        total_loan_amount=("total_loan_amount", "sum"),
        total_transactions=("total_transactions", "sum"),
        transaction_value=("total_transaction_value", "sum"),
        avg_balance=("total_balance", "mean"),
        avg_income=("annual_income", "mean"),
        avg_credit_score=("credit_score", "mean"),
        avg_products=("products_number", "mean"),
        avg_service_interactions=("total_service_interactions", "mean")
    )
    .reset_index()
)
segment_kpis.round(2)

,segment_name,customers,total_balance,total_loan_amount,total_transactions,transaction_value,avg_balance,avg_income,avg_credit_score,avg_products,avg_service_interactions
0,Core Banking Customers,2295,2.517168e+08,1.563396e+08,45813.0,13504000.61,109680.50,67067.71,649.83,2.29,1.18
1,High-Service Customers,1167,1.269212e+08,9.553669e+07,23232.0,6934574.15,108758.55,66526.05,654.71,2.33,3.66
2,High-Value Loan Customers,410,5.091761e+07,4.848854e+08,8279.0,2449532.02,124189.28,63246.59,652.46,3.56,1.97
3,Low Engagement Customers,1128,1.195545e+05,0.000000e+00,83.0,18356.43,105.99,65032.89,655.84,0.50,0.01


In [12]:
# Verify the KPI table
print("Segment KPI columns:")
print("-" * 40)

print(segment_kpis.columns.tolist())

print("\nShape:")
print(segment_kpis.shape)

print("\nKPI table:")
print(segment_kpis.to_string(index=False))

Segment KPI columns:
----------------------------------------
['segment_name', 'customers', 'total_balance', 'total_loan_amount', 'total_transactions', 'transaction_value', 'avg_balance', 'avg_income', 'avg_credit_score', 'avg_products', 'avg_service_interactions']

Shape:
(4, 11)

KPI table:
             segment_name  customers  total_balance  total_loan_amount  total_transactions  transaction_value   avg_balance   avg_income  avg_credit_score  avg_products  avg_service_interactions
   Core Banking Customers       2295   251716755.62       156339647.40             45813.0        13504000.61 109680.503538 67067.712418        649.827887      2.290196                  1.181264
   High-Service Customers       1167   126921232.37        95536688.40             23232.0         6934574.15 108758.553873 66526.049700        654.713796      2.331620                  3.664096
High-Value Loan Customers        410    50917606.08       484885353.81              8279.0         2449532.02 124189.2831

In [13]:
# Segment Performance
segment_kpis["customer_pct"] = (
    segment_kpis["customers"] / total_customers * 100
)

segment_kpis["balance_pct"] = (
    segment_kpis["total_balance"] / total_deposits * 100
)

segment_kpis["loan_pct"] = (
    segment_kpis["total_loan_amount"] / total_loan_amount * 100
)

segment_kpis["transaction_pct"] = (
    segment_kpis["transaction_value"] / total_transaction_value * 100
)

In [14]:
segment_kpis[
    [
        "segment_name",
        "customers",
        "customer_pct",
        "balance_pct",
        "loan_pct",
        "transaction_pct"
    ]
].round(2)

,segment_name,customers,customer_pct,balance_pct,loan_pct,transaction_pct
0,Core Banking Customers,2295,45.90,58.58,21.22,58.95
1,High-Service Customers,1167,23.34,29.54,12.97,30.27
2,High-Value Loan Customers,410,8.20,11.85,65.81,10.69
3,Low Engagement Customers,1128,22.56,0.03,0.00,0.08


In [15]:
# Key Segment Findings
largest_segment = segment_kpis.loc[
    segment_kpis["customers"].idxmax(), "segment_name"
]

highest_balance_segment = segment_kpis.loc[
    segment_kpis["total_balance"].idxmax(), "segment_name"
]

highest_loan_segment = segment_kpis.loc[
    segment_kpis["total_loan_amount"].idxmax(), "segment_name"
]

highest_transaction_segment = segment_kpis.loc[
    segment_kpis["transaction_value"].idxmax(), "segment_name"
]

lowest_engagement_segment = segment_kpis.loc[
    segment_kpis["avg_products"].idxmin(), "segment_name"
]

In [16]:
print("KEY SEGMENT FINDINGS")
print("=" * 40)

print("Largest segment:", largest_segment)
print("Highest balance:", highest_balance_segment)
print("Highest loan exposure:", highest_loan_segment)
print("Highest transaction value:", highest_transaction_segment)
print("Lowest product engagement:", lowest_engagement_segment)

KEY SEGMENT FINDINGS
Largest segment: Core Banking Customers
Highest balance: Core Banking Customers
Highest loan exposure: High-Value Loan Customers
Highest transaction value: Core Banking Customers
Lowest product engagement: Low Engagement Customers


In [17]:
# Create Business Insights
low_engagement = segment_kpis[
    segment_kpis["segment_name"] == "Low Engagement Customers"
].iloc[0]

high_loan = segment_kpis[
    segment_kpis["segment_name"] == "High-Value Loan Customers"
].iloc[0]

core = segment_kpis[
    segment_kpis["segment_name"] == "Core Banking Customers"
].iloc[0]

In [18]:
print("BUSINESS INSIGHTS")
print("=" * 60)

print(
    f"1. Core Banking Customers are the largest segment, "
    f"representing {core['customer_pct']:.2f}% of customers."
)

print(
    f"2. Core Banking Customers hold {core['balance_pct']:.2f}% "
    f"of the total customer balance."
)

print(
    f"3. High-Value Loan Customers represent only "
    f"{high_loan['customer_pct']:.2f}% of customers but account for "
    f"{high_loan['loan_pct']:.2f}% of total loan exposure."
)

print(
    f"4. Low Engagement Customers represent "
    f"{low_engagement['customer_pct']:.2f}% of customers "
    f"but contribute only {low_engagement['balance_pct']:.2f}% "
    f"of total customer balance."
)

print(
    f"5. Core Banking Customers generate the highest transaction "
    f"value, contributing {core['transaction_pct']:.2f}% of total transaction value."
)

BUSINESS INSIGHTS
1. Core Banking Customers are the largest segment, representing 45.90% of customers.
2. Core Banking Customers hold 58.58% of the total customer balance.
3. High-Value Loan Customers represent only 8.20% of customers but account for 65.81% of total loan exposure.
4. Low Engagement Customers represent 22.56% of customers but contribute only 0.03% of total customer balance.
5. Core Banking Customers generate the highest transaction value, contributing 58.95% of total transaction value.


In [19]:
# Business Recommendations
print("BUSINESS RECOMMENDATIONS")
print("=" * 60)

print("1. Core Banking Customers")
print("   Focus on retention, cross-selling, and increasing product adoption.")

print("\n2. High-Value Loan Customers")
print("   Prioritize loan relationship management, credit monitoring, and")
print("   targeted financial products because this segment holds most loan exposure.")

print("\n3. Low Engagement Customers")
print("   Launch targeted engagement campaigns, digital banking activation,")
print("   and personalized offers to increase activity and product usage.")

print("\n4. High-Service Customers")
print("   Improve service efficiency and identify opportunities to migrate")
print("   customers toward more digital self-service channels.")

BUSINESS RECOMMENDATIONS
1. Core Banking Customers
   Focus on retention, cross-selling, and increasing product adoption.

2. High-Value Loan Customers
   Prioritize loan relationship management, credit monitoring, and
   targeted financial products because this segment holds most loan exposure.

3. Low Engagement Customers
   Launch targeted engagement campaigns, digital banking activation,
   and personalized offers to increase activity and product usage.

4. High-Service Customers
   Improve service efficiency and identify opportunities to migrate
   customers toward more digital self-service channels.


In [20]:
# Find KPI Summary
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Total Customer Balance",
        "Total Loan Amount",
        "Total Transactions",
        "Total Transaction Value",
        "Average Customer Balance",
        "Average Loan Amount",
        "Average Transaction Value",
        "Loan Penetration",
        "Active Customer Rate"
    ],
    
    "Value": [
        total_customers,
        total_deposits,
        total_loan_amount,
        total_transactions,
        total_transaction_value,
        average_balance,
        average_loan_amount,
        average_transaction_value,
        loan_penetration,
        active_customer_rate
    ]
})

kpi_summary

,KPI,Value
0,Total Customers,5.000000e+03
1,Total Customer Balance,4.296751e+08
2,Total Loan Amount,7.367617e+08
3,Total Transactions,7.740700e+04
4,Total Transaction Value,2.290646e+07
5,Average Customer Balance,8.593503e+04
6,Average Loan Amount,4.850307e+05
7,Average Transaction Value,2.293995e+02
8,Loan Penetration,3.038000e+01
9,Active Customer Rate,7.288000e+01


In [21]:
# Format KPI Summary
kpi_summary["Value"] = kpi_summary["Value"].round(2)

kpi_summary

,KPI,Value
0,Total Customers,5.000000e+03
1,Total Customer Balance,4.296751e+08
2,Total Loan Amount,7.367617e+08
3,Total Transactions,7.740700e+04
4,Total Transaction Value,2.290646e+07
5,Average Customer Balance,8.593503e+04
6,Average Loan Amount,4.850307e+05
7,Average Transaction Value,2.294000e+02
8,Loan Penetration,3.038000e+01
9,Active Customer Rate,7.288000e+01


In [22]:
segment_summary = segment_kpis[
    [
        "segment_name",
        "customers",
        "customer_pct",
        "total_balance",
        "balance_pct",
        "total_loan_amount",
        "loan_pct",
        "transaction_value",
        "transaction_pct"
    ]
].copy()

segment_summary

,segment_name,customers,customer_pct,total_balance,balance_pct,total_loan_amount,loan_pct,transaction_value,transaction_pct
0,Core Banking Customers,2295,45.90,2.517168e+08,58.583038,1.563396e+08,21.219839,13504000.61,58.952796
1,High-Service Customers,1167,23.34,1.269212e+08,29.538881,9.553669e+07,12.967109,6934574.15,30.273439
2,High-Value Loan Customers,410,8.20,5.091761e+07,11.850256,4.848854e+08,65.813052,2449532.02,10.693628
3,Low Engagement Customers,1128,22.56,1.195545e+05,0.027824,0.000000e+00,0.000000,18356.43,0.080136


In [23]:
# Export Analysis Results
import os

os.makedirs("../outputs/exports", exist_ok=True)

print("Export folder ready")

Export folder ready


In [24]:
kpi_summary.to_csv(
    "../outputs/exports/kpi_summary.csv",
    index=False
)

print("KPI summary exported")

KPI summary exported


In [25]:
segment_summary.to_csv(
    "../outputs/exports/segment_summary.csv",
    index=False
)

print("Segment summary exported")

Segment summary exported


In [26]:
with pd.ExcelWriter("../outputs/exports/business_analysis.xlsx") as writer:
    kpi_summary.to_excel(
        writer,
        sheet_name="KPI Summary",
        index=False
    )
    
    segment_summary.to_excel(
        writer,
        sheet_name="Segment Summary",
        index=False
    )

print("Excel report exported successfully")

Excel report exported successfully


In [27]:
# Final Validation
import os

export_files = os.listdir("../outputs/exports")

print("BUSINESS ANALYSIS EXPORTS")
print("=" * 40)

for file in export_files:
    print(file)

BUSINESS ANALYSIS EXPORTS
business_analysis.xlsx
kpi_summary.csv
segment_summary.csv


In [30]:
print("Business Analysis")
print("=" * 40)

print("Total Customers:", total_customers)
print("Loan Penetration:", f"{loan_penetration:.2f}%")
print("Active Customer Rate:", f"{active_customer_rate:.2f}%")
print("Segments:", customer_360_segmented["segment_name"].nunique())

print("\nKey Segments:")
for segment in segment_kpis["segment_name"]:
    print("-", segment)

print("\nExports:")
print("- kpi_summary.csv")
print("- segment_summary.csv")
print("- business_analysis.xlsx")

Business Analysis
Total Customers: 5000
Loan Penetration: 30.38%
Active Customer Rate: 72.88%
Segments: 4

Key Segments:
- Core Banking Customers
- High-Service Customers
- High-Value Loan Customers
- Low Engagement Customers

Exports:
- kpi_summary.csv
- segment_summary.csv
- business_analysis.xlsx
